# CreditWise Loan System - ML Analysis

**Objective:** Build a machine learning model to predict loan approval (Approved/Rejected) based on applicant details.

**Dataset:** 1000 loan applications with 20 features covering personal, financial, and credit information.

## 1. Data Loading & Initial Exploration

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100

In [2]:
df = pd.read_csv('../loan_approval_data.csv')
print(f"Dataset Shape: {df.shape}")
print(f"\nRows: {df.shape[0]}, Columns: {df.shape[1]}")
df.head()

Dataset Shape: (1000, 20)

Rows: 1000, Columns: 20


,Applicant_ID,Applicant_Income,Coapplicant_Income,Employment_Status,Age,Marital_Status,Dependents,Credit_Score,Existing_Loans,DTI_Ratio,Savings,Collateral_Value,Loan_Amount,Loan_Term,Loan_Purpose,Property_Area,Education_Level,Gender,Employer_Category,Loan_Approved
0,1.0,17795.0,1387.0,Salaried,51.0,Married,0.0,637.0,4.0,0.53,19403.0,45638.0,16619.0,84.0,Personal,Urban,Not Graduate,Female,Private,No
1,2.0,2860.0,2679.0,Salaried,46.0,Married,3.0,621.0,2.0,0.30,2580.0,49272.0,38687.0,NaN,Car,Semiurban,Graduate,NaN,Private,No
2,3.0,7390.0,2106.0,Salaried,25.0,Single,2.0,674.0,4.0,0.20,13844.0,6908.0,27943.0,72.0,NaN,Urban,NaN,Female,Government,Yes
3,4.0,13964.0,8173.0,Salaried,40.0,Married,2.0,579.0,3.0,0.31,9553.0,10844.0,27819.0,60.0,Business,Rural,Graduate,Female,Government,No
4,5.0,13284.0,4223.0,Self-employed,31.0,Single,2.0,721.0,1.0,0.29,9386.0,37629.0,12741.0,72.0,Car,NaN,Graduate,Male,Private,Yes


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 20 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Applicant_ID        950 non-null    float64
 1   Applicant_Income    950 non-null    float64
 2   Coapplicant_Income  950 non-null    float64
 3   Employment_Status   950 non-null    str    
 4   Age                 950 non-null    float64
 5   Marital_Status      950 non-null    str    
 6   Dependents          950 non-null    float64
 7   Credit_Score        950 non-null    float64
 8   Existing_Loans      950 non-null    float64
 9   DTI_Ratio           950 non-null    float64
 10  Savings             950 non-null    float64
 11  Collateral_Value    950 non-null    float64
 12  Loan_Amount         950 non-null    float64
 13  Loan_Term           950 non-null    float64
 14  Loan_Purpose        950 non-null    str    
 15  Property_Area       950 non-null    str    
 16  Education_Level   

In [4]:
df.describe()

,Applicant_ID,Applicant_Income,Coapplicant_Income,Age,Dependents,Credit_Score,Existing_Loans,DTI_Ratio,Savings,Collateral_Value,Loan_Amount,Loan_Term
count,950.000000,950.000000,950.000000,950.000000,950.000000,950.000000,950.000000,950.000000,950.000000,950.000000,950.000000,950.000000
mean,501.220000,10852.571579,5082.455789,39.971579,1.474737,676.033684,1.950526,0.347263,9940.452632,24802.792632,20522.825263,48.000000
std,289.608451,5061.632859,2943.161570,11.139797,1.105067,71.346015,1.406246,0.144341,5860.736885,14345.696031,11504.142575,24.245322
min,1.000000,2009.000000,1.000000,21.000000,0.000000,550.000000,0.000000,0.100000,65.000000,36.000000,1015.000000,12.000000
25%,250.250000,6730.750000,2472.750000,30.250000,1.000000,616.250000,1.000000,0.220000,4760.250000,12698.250000,9806.250000,24.000000
50%,499.500000,10548.000000,5205.500000,40.000000,1.000000,678.000000,2.000000,0.340000,9880.500000,24321.000000,21210.500000,48.000000
75%,752.750000,15190.000000,7620.750000,49.000000,2.000000,737.000000,3.000000,0.480000,15074.500000,36947.000000,30263.000000,72.000000
max,1000.000000,19988.000000,9996.000000,59.000000,3.000000,799.000000,4.000000,0.600000,19996.000000,49954.000000,39995.000000,84.000000


In [5]:
df.describe(include='object')

,Employment_Status,Marital_Status,Loan_Purpose,Property_Area,Education_Level,Gender,Employer_Category,Loan_Approved
count,950,950,950,950,950,950,950,950
unique,4,2,5,3,2,2,5,2
top,Salaried,Married,Business,Urban,Graduate,Male,Private,No
freq,465,593,202,467,672,571,372,652


## 2. Handle Missing Values & Duplicates

### 2.1 Check Missing Values

In [6]:
missing = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Missing %': (df.isnull().mean() * 100).round(2)
})
missing[missing['Missing Count'] > 0].sort_values('Missing %', ascending=False)

,Missing Count,Missing %
Applicant_ID,50,5.0
Applicant_Income,50,5.0
Employer_Category,50,5.0
Gender,50,5.0
Education_Level,50,5.0
Property_Area,50,5.0
Loan_Purpose,50,5.0
Loan_Term,50,5.0
Loan_Amount,50,5.0
Collateral_Value,50,5.0


In [7]:
# Visualize missing values
plt.figure(figsize=(12, 5))
missing_cols = df.columns[df.isnull().any()]
sns.barplot(x=missing_cols, y=df[missing_cols].isnull().sum(), palette='Reds_r')
plt.title('Missing Values Count by Column')
plt.xticks(rotation=45, ha='right')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

### 2.2 Handle Missing Values

In [8]:
# Separate numerical and categorical columns
num_cols = df.select_dtypes(include='number').columns.tolist()
cat_cols = df.select_dtypes(include=['object', 'string']).columns.tolist()

print("Numerical columns:", num_cols)
print("\nCategorical columns:", cat_cols)

Numerical columns: ['Applicant_ID', 'Applicant_Income', 'Coapplicant_Income', 'Age', 'Dependents', 'Credit_Score', 'Existing_Loans', 'DTI_Ratio', 'Savings', 'Collateral_Value', 'Loan_Amount', 'Loan_Term']

Categorical columns: ['Employment_Status', 'Marital_Status', 'Loan_Purpose', 'Property_Area', 'Education_Level', 'Gender', 'Employer_Category', 'Loan_Approved']


In [9]:
# Fill numerical columns with median (Pandas 3.x compatible)
for col in num_cols:
    if df[col].isnull().sum() > 0:
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
        print(f"Filled '{col}' with median: {median_val}")

# Fill categorical columns with mode
for col in cat_cols:
    if df[col].isnull().sum() > 0:
        mode_val = df[col].mode()[0]
        df[col] = df[col].fillna(mode_val)
        print(f"Filled '{col}' with mode: {mode_val}")

print(f"\nTotal missing values after handling: {df.isnull().sum().sum()}")

Filled 'Applicant_ID' with median: 499.5
Filled 'Applicant_Income' with median: 10548.0
Filled 'Coapplicant_Income' with median: 5205.5
Filled 'Age' with median: 40.0
Filled 'Dependents' with median: 1.0
Filled 'Credit_Score' with median: 678.0
Filled 'Existing_Loans' with median: 2.0
Filled 'DTI_Ratio' with median: 0.34
Filled 'Savings' with median: 9880.5
Filled 'Collateral_Value' with median: 24321.0
Filled 'Loan_Amount' with median: 21210.5
Filled 'Loan_Term' with median: 48.0
Filled 'Employment_Status' with mode: Salaried
Filled 'Marital_Status' with mode: Married
Filled 'Loan_Purpose' with mode: Business
Filled 'Property_Area' with mode: Urban
Filled 'Education_Level' with mode: Graduate
Filled 'Gender' with mode: Male
Filled 'Employer_Category' with mode: Private
Filled 'Loan_Approved' with mode: No

Total missing values after handling: 0


### 2.3 Handle Duplicates

In [10]:
print(f"Duplicate rows: {df.duplicated().sum()}")
df = df.drop_duplicates()
print(f"Shape after removing duplicates: {df.shape}")

Duplicate rows: 0
Shape after removing duplicates: (1000, 20)


In [11]:
# Drop Applicant_ID (identifier, not a feature)
df = df.drop('Applicant_ID', axis=1)
print(f"Shape after dropping ID: {df.shape}")

# Drop rows where target is missing
before = len(df)
df = df.dropna(subset=['Loan_Approved'])
print(f"Dropped {before - len(df)} rows with missing target")
print(f"Final shape: {df.shape}")

Shape after dropping ID: (1000, 19)
Dropped 0 rows with missing target
Final shape: (1000, 19)


## 3. Exploratory Data Analysis (EDA)

### 3.1 Target Variable Distribution

In [12]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

target_counts = df['Loan_Approved'].value_counts()
colors = ['#e74c3c', '#2ecc71']
axes[0].bar(target_counts.index, target_counts.values, color=colors)
axes[0].set_title('Loan Approval Distribution')
axes[0].set_xlabel('Loan Approved')
axes[0].set_ylabel('Count')
for i, v in enumerate(target_counts.values):
    axes[0].text(i, v + 10, str(v), ha='center', fontweight='bold')

axes[1].pie(target_counts.values, labels=target_counts.index, autopct='%1.1f%%',
            colors=colors, startangle=90, explode=(0.05, 0.05))
axes[1].set_title('Loan Approval Ratio')

plt.tight_layout()
plt.show()

print(f"Class distribution:\n{target_counts}")

Class distribution:
Loan_Approved
No     702
Yes    298
Name: count, dtype: int64


### 3.2 Numerical Features Distribution

In [13]:
num_features = [c for c in df.select_dtypes(include='number').columns if c not in ['Applicant_ID']]
fig, axes = plt.subplots(3, 4, figsize=(20, 15))
axes = axes.flatten()

for i, col in enumerate(num_features):
    if i < len(axes):
        sns.histplot(data=df, x=col, hue='Loan_Approved', kde=True, ax=axes[i], palette=colors)
        axes[i].set_title(f'{col} Distribution')

for j in range(len(num_features), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

### 3.3 Categorical Features vs Target

In [14]:
cat_features = [c for c in df.select_dtypes(include=['object', 'string']).columns if c != 'Loan_Approved']
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, col in enumerate(cat_features):
    if i < len(axes):
        ct = pd.crosstab(df[col], df['Loan_Approved'])
        ct.plot(kind='bar', ax=axes[i], color=colors, rot=45)
        axes[i].set_title(f'{col} vs Loan Approval')
        axes[i].legend(title='Approved')

for j in range(len(cat_features), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

### 3.4 Boxplots - Outlier Detection

In [15]:
key_numerical = ['Applicant_Income', 'Coapplicant_Income', 'Loan_Amount',
                  'Credit_Score', 'DTI_Ratio', 'Savings', 'Collateral_Value']

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, col in enumerate(key_numerical):
    sns.boxplot(x='Loan_Approved', y=col, data=df, ax=axes[i], palette=colors)
    axes[i].set_title(f'{col}')

axes[-1].set_visible(False)
plt.suptitle('Boxplots: Numerical Features by Loan Approval', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 4. Feature Encoding

In [16]:
# Encode target variable
df['Loan_Approved'] = df['Loan_Approved'].map({'Yes': 1, 'No': 0})
print(f"Target encoding: Yes->1, No->0")
print(f"Value counts after encoding:\n{df['Loan_Approved'].value_counts()}")

Target encoding: Yes->1, No->0
Value counts after encoding:
Loan_Approved
0    702
1    298
Name: count, dtype: int64


In [17]:
from sklearn.preprocessing import LabelEncoder

# Label Encoding for binary columns
le = LabelEncoder()
binary_cols = ['Gender', 'Marital_Status']

for col in binary_cols:
    df[col] = le.fit_transform(df[col].astype(str))
    print(f"Label Encoded '{col}': {dict(zip(le.classes_, le.transform(le.classes_)))}")

Label Encoded 'Gender': {'Female': np.int64(0), 'Male': np.int64(1)}
Label Encoded 'Marital_Status': {'Married': np.int64(0), 'Single': np.int64(1)}


In [18]:
# One-Hot Encoding for nominal (multi-category) columns
nominal_cols = ['Employment_Status', 'Loan_Purpose', 'Property_Area',
                'Education_Level', 'Employer_Category']

print(f"Shape before One-Hot Encoding: {df.shape}")
df = pd.get_dummies(df, columns=nominal_cols, drop_first=True)
# Convert bool columns to int
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)
print(f"Shape after One-Hot Encoding: {df.shape}")
print(f"\nNew columns: {[c for c in df.columns if any(n in c for n in nominal_cols)]}")

Shape before One-Hot Encoding: (1000, 19)
Shape after One-Hot Encoding: (1000, 28)

New columns: ['Employment_Status_Salaried', 'Employment_Status_Self-employed', 'Employment_Status_Unemployed', 'Loan_Purpose_Car', 'Loan_Purpose_Education', 'Loan_Purpose_Home', 'Loan_Purpose_Personal', 'Property_Area_Semiurban', 'Property_Area_Urban', 'Education_Level_Not Graduate', 'Employer_Category_Government', 'Employer_Category_MNC', 'Employer_Category_Private', 'Employer_Category_Unemployed']


## 5. Correlation Heatmap

In [19]:
plt.figure(figsize=(18, 14))
correlation = df.corr(numeric_only=True)
mask = np.triu(np.ones_like(correlation, dtype=bool))
sns.heatmap(correlation, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, square=True,
            cbar_kws={'shrink': 0.8})
plt.title('Feature Correlation Heatmap', fontsize=16, pad=20)
plt.tight_layout()
plt.show()

In [20]:
# Top features correlated with target
target_corr = correlation['Loan_Approved'].drop('Loan_Approved').sort_values(ascending=False)
print("Top Positive Correlations with Loan_Approved:")
print(target_corr.head(10).to_string())
print("\nTop Negative Correlations with Loan_Approved:")
print(target_corr.tail(10).to_string())

Top Positive Correlations with Loan_Approved:
Credit_Score                       0.451235
Applicant_Income                   0.120177
Employer_Category_MNC              0.069049
Loan_Purpose_Personal              0.034043
Marital_Status                     0.030182
Property_Area_Urban                0.025963
Collateral_Value                   0.021709
Coapplicant_Income                 0.003957
Loan_Purpose_Home                  0.002118
Employment_Status_Self-employed   -0.001337

Top Negative Correlations with Loan_Approved:
Existing_Loans                 -0.035101
Employer_Category_Government   -0.039187
Employment_Status_Salaried     -0.041428
Employment_Status_Unemployed   -0.044464
Education_Level_Not Graduate   -0.052920
Gender                         -0.054342
Loan_Purpose_Car               -0.056416
Loan_Term                      -0.086644
Loan_Amount                    -0.126608
DTI_Ratio                      -0.444766


## 6. Feature Scaling

In [21]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

X = df.drop('Loan_Approved', axis=1)
y = df['Loan_Approved']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Target distribution:\n{y.value_counts()}")

Features shape: (1000, 27)
Target shape: (1000,)
Target distribution:
Loan_Approved
0    702
1    298
Name: count, dtype: int64


In [22]:
# Train-Test Split BEFORE scaling (to avoid data leakage)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"\nTraining target distribution:\n{y_train.value_counts()}")
print(f"\nTest target distribution:\n{y_test.value_counts()}")

Training set: (800, 27)
Test set: (200, 27)

Training target distribution:
Loan_Approved
0    562
1    238
Name: count, dtype: int64

Test target distribution:
Loan_Approved
0    140
1     60
Name: count, dtype: int64


In [23]:
# Scale features - fit on train, transform both
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

print("Scaling complete!")
print(f"\nTrain mean (should be ~0): {X_train_scaled.mean().mean():.6f}")
print(f"Train std (should be ~1): {X_train_scaled.std().mean():.6f}")

Scaling complete!

Train mean (should be ~0): -0.000000
Train std (should be ~1): 1.000626


## 7. Model Training & Evaluation

In [24]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report,
                             roc_auc_score, roc_curve)

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(probability=True, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5)
}

results = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]

    results[name] = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1 Score': f1_score(y_test, y_pred),
        'ROC AUC': roc_auc_score(y_test, y_prob)
    }
    print(f"Trained: {name}")

results_df = pd.DataFrame(results).T.round(4)
results_df.sort_values('F1 Score', ascending=False)

Trained: Logistic Regression


Trained: Random Forest
Trained: SVM
Trained: Decision Tree
Trained: KNN


,Accuracy,Precision,Recall,F1 Score,ROC AUC
Random Forest,0.895,0.7826,0.9000,0.8372,0.9342
Decision Tree,0.875,0.7869,0.8000,0.7934,0.8536
SVM,0.815,0.6716,0.7500,0.7087,0.8927
Logistic Regression,0.805,0.6479,0.7667,0.7023,0.8886
KNN,0.745,0.5738,0.5833,0.5785,0.7527


### 7.1 Model Comparison Visualization

In [25]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

results_df.plot(kind='bar', ax=axes[0], colormap='Set2')
axes[0].set_title('Model Performance Comparison')
axes[0].set_ylabel('Score')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right')
axes[0].legend(loc='lower right')
axes[0].set_ylim(0, 1.1)

for name, model in models.items():
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = results[name]['ROC AUC']
    axes[1].plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', linewidth=2)

axes[1].plot([0, 1], [0, 1], '--', color='gray', linewidth=1)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curves')
axes[1].legend(loc='lower right')

plt.tight_layout()
plt.show()

### 7.2 Confusion Matrix (Best Model)

In [26]:
best_name = results_df['F1 Score'].idxmax()
best_model = models[best_name]
print(f"Best Model: {best_name}")

y_pred_best = best_model.predict(X_test_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm = confusion_matrix(y_test, y_pred_best)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Rejected', 'Approved'],
            yticklabels=['Rejected', 'Approved'])
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].set_title(f'Confusion Matrix - {best_name}')

cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues', ax=axes[1],
            xticklabels=['Rejected', 'Approved'],
            yticklabels=['Rejected', 'Approved'])
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')
axes[1].set_title(f'Normalized Confusion Matrix - {best_name}')

plt.tight_layout()
plt.show()

print(f"\nClassification Report:\n")
print(classification_report(y_test, y_pred_best, target_names=['Rejected', 'Approved']))

Best Model: Random Forest



Classification Report:



              precision    recall  f1-score   support

    Rejected       0.95      0.89      0.92       140
    Approved       0.78      0.90      0.84        60

    accuracy                           0.90       200
   macro avg       0.87      0.90      0.88       200
weighted avg       0.90      0.90      0.90       200



## 8. Feature Engineering (Improve Model)

### 8.1 Create New Features

In [27]:
# Reload original data for feature engineering
df_fe = pd.read_csv('../loan_approval_data.csv')

# Handle missing values (Pandas 3.x compatible)
for col in df_fe.select_dtypes(include='number').columns:
    df_fe[col] = df_fe[col].fillna(df_fe[col].median())
for col in df_fe.select_dtypes(include=['object', 'string']).columns:
    df_fe[col] = df_fe[col].fillna(df_fe[col].mode()[0])

df_fe = df_fe.drop_duplicates()
df_fe = df_fe.drop('Applicant_ID', axis=1)
df_fe = df_fe.dropna(subset=['Loan_Approved'])

# ── New Feature Engineering ──

# 1. Total Income
df_fe['Total_Income'] = df_fe['Applicant_Income'] + df_fe['Coapplicant_Income']

# 2. Income to Loan Ratio
df_fe['Income_Loan_Ratio'] = df_fe['Total_Income'] / (df_fe['Loan_Amount'] + 1)

# 3. EMI Estimate
df_fe['EMI_Estimate'] = df_fe['Loan_Amount'] / (df_fe['Loan_Term'] + 1)

# 4. EMI to Income Ratio
df_fe['EMI_Income_Ratio'] = df_fe['EMI_Estimate'] / (df_fe['Total_Income'] + 1)

# 5. Collateral Coverage
df_fe['Collateral_Coverage'] = df_fe['Collateral_Value'] / (df_fe['Loan_Amount'] + 1)

# 6. Savings to Loan Ratio
df_fe['Savings_Loan_Ratio'] = df_fe['Savings'] / (df_fe['Loan_Amount'] + 1)

# 7. Credit Risk Flag
df_fe['Low_Credit'] = (df_fe['Credit_Score'] < 600).astype(int)

# 8. High DTI Flag
df_fe['High_DTI'] = (df_fe['DTI_Ratio'] > 0.5).astype(int)

# 9. Log transformations for skewed features
df_fe['Log_Income'] = np.log1p(df_fe['Applicant_Income'])
df_fe['Log_LoanAmount'] = np.log1p(df_fe['Loan_Amount'])

print("New features created:")
new_features = ['Total_Income', 'Income_Loan_Ratio', 'EMI_Estimate', 'EMI_Income_Ratio',
                'Collateral_Coverage', 'Savings_Loan_Ratio', 'Low_Credit', 'High_DTI',
                'Log_Income', 'Log_LoanAmount']
for f in new_features:
    print(f"  - {f}")

New features created:
  - Total_Income
  - Income_Loan_Ratio
  - EMI_Estimate
  - EMI_Income_Ratio
  - Collateral_Coverage
  - Savings_Loan_Ratio
  - Low_Credit
  - High_DTI
  - Log_Income
  - Log_LoanAmount


### 8.2 Encode & Scale with New Features

In [28]:
# Encode target
df_fe['Loan_Approved'] = df_fe['Loan_Approved'].map({'Yes': 1, 'No': 0})

# Label encode binary
for col in ['Gender', 'Marital_Status']:
    df_fe[col] = LabelEncoder().fit_transform(df_fe[col].astype(str))

# One-Hot encode nominal
nominal_cols = ['Employment_Status', 'Loan_Purpose', 'Property_Area',
                'Education_Level', 'Employer_Category']
df_fe = pd.get_dummies(df_fe, columns=nominal_cols, drop_first=True)
bool_cols = df_fe.select_dtypes(include='bool').columns
df_fe[bool_cols] = df_fe[bool_cols].astype(int)

# Split
X_fe = df_fe.drop('Loan_Approved', axis=1)
y_fe = df_fe['Loan_Approved']

X_train_fe, X_test_fe, y_train_fe, y_test_fe = train_test_split(
    X_fe, y_fe, test_size=0.2, random_state=42, stratify=y_fe
)

# Scale
scaler_fe = StandardScaler()
X_train_fe_scaled = pd.DataFrame(scaler_fe.fit_transform(X_train_fe), columns=X_train_fe.columns)
X_test_fe_scaled = pd.DataFrame(scaler_fe.transform(X_test_fe), columns=X_test_fe.columns)

print(f"Features with engineering: {X_fe.shape[1]} (was {X.shape[1]})")

Features with engineering: 37 (was 27)


### 8.3 Retrain & Compare

In [29]:
results_fe = {}
models_fe = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(probability=True, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5)
}

for name, model in models_fe.items():
    model.fit(X_train_fe_scaled, y_train_fe)
    y_pred = model.predict(X_test_fe_scaled)
    y_prob = model.predict_proba(X_test_fe_scaled)[:, 1]

    results_fe[name] = {
        'Accuracy': accuracy_score(y_test_fe, y_pred),
        'Precision': precision_score(y_test_fe, y_pred),
        'Recall': recall_score(y_test_fe, y_pred),
        'F1 Score': f1_score(y_test_fe, y_pred),
        'ROC AUC': roc_auc_score(y_test_fe, y_prob)
    }

results_fe_df = pd.DataFrame(results_fe).T.round(4)

print("=" * 60)
print("BEFORE Feature Engineering:")
print("=" * 60)
print(results_df.to_string())
print()
print("=" * 60)
print("AFTER Feature Engineering:")
print("=" * 60)
print(results_fe_df.to_string())
print()

print("=" * 60)
print("IMPROVEMENT (F1 Score):")
print("=" * 60)
for name in results_df.index:
    before = results_df.loc[name, 'F1 Score']
    after = results_fe_df.loc[name, 'F1 Score']
    diff = after - before
    symbol = '+' if diff > 0 else ('-' if diff < 0 else '=')
    print(f"  {name}: {before:.4f} -> {after:.4f} ({symbol}{abs(diff):.4f})")

BEFORE Feature Engineering:
                     Accuracy  Precision  Recall  F1 Score  ROC AUC
Logistic Regression     0.805     0.6479  0.7667    0.7023   0.8886
Random Forest           0.895     0.7826  0.9000    0.8372   0.9342
SVM                     0.815     0.6716  0.7500    0.7087   0.8927
Decision Tree           0.875     0.7869  0.8000    0.7934   0.8536
KNN                     0.745     0.5738  0.5833    0.5785   0.7527

AFTER Feature Engineering:
                     Accuracy  Precision  Recall  F1 Score  ROC AUC
Logistic Regression     0.815     0.6620  0.7833    0.7176   0.9077
Random Forest           0.910     0.8000  0.9333    0.8615   0.9389
SVM                     0.790     0.6452  0.6667    0.6557   0.8964
Decision Tree           0.875     0.7692  0.8333    0.8000   0.8631
KNN                     0.765     0.5942  0.6833    0.6357   0.7990

IMPROVEMENT (F1 Score):
  Logistic Regression: 0.7023 -> 0.7176 (+0.0153)
  Random Forest: 0.8372 -> 0.8615 (+0.0243)
  SVM: 0.

### 8.4 Feature Importance

In [30]:
best_fe_name = results_fe_df['F1 Score'].idxmax()
best_fe_model = models_fe[best_fe_name]

if hasattr(best_fe_model, 'feature_importances_'):
    importances = best_fe_model.feature_importances_
    feat_imp = pd.Series(importances, index=X_fe.columns).sort_values(ascending=True)

    plt.figure(figsize=(10, 10))
    feat_imp.tail(20).plot(kind='barh', color='steelblue')
    plt.title(f'Top 20 Feature Importances ({best_fe_name})')
    plt.xlabel('Importance')
    plt.tight_layout()
    plt.show()

    print("\nTop 10 Most Important Features:")
    for i, (feat, imp) in enumerate(feat_imp.tail(10).iloc[::-1].items(), 1):
        print(f"  {i}. {feat}: {imp:.4f}")


Top 10 Most Important Features:
  1. Credit_Score: 0.2584
  2. DTI_Ratio: 0.2365
  3. High_DTI: 0.0470
  4. Log_Income: 0.0459
  5. Income_Loan_Ratio: 0.0452
  6. Low_Credit: 0.0445
  7. Applicant_Income: 0.0378
  8. Collateral_Coverage: 0.0254
  9. EMI_Income_Ratio: 0.0242
  10. Total_Income: 0.0232


## 9. Save Best Model for Deployment

In [31]:
import joblib
import os

os.makedirs('../backend/model', exist_ok=True)

best_final_name = results_fe_df['F1 Score'].idxmax()
best_final_model = models_fe[best_final_name]

joblib.dump(best_final_model, '../backend/model/loan_model.pkl')
joblib.dump(scaler_fe, '../backend/model/scaler.pkl')

# Save feature columns for backend reference
feature_cols = X_fe.columns.tolist()
joblib.dump(feature_cols, '../backend/model/feature_columns.pkl')

print(f"Saved model: {best_final_name}")
print(f"Saved scaler: StandardScaler")
print(f"Saved feature columns: {len(feature_cols)} features")
print(f"\nFiles saved in: ../backend/model/")

Saved model: Random Forest
Saved scaler: StandardScaler
Saved feature columns: 37 features

Files saved in: ../backend/model/


## Summary

| Step | Task | Status |
|------|------|--------|
| 1 | Data Loading | Done |
| 2 | Missing Values & Duplicates | Done |
| 3 | Exploratory Data Analysis | Done |
| 4 | Feature Encoding | Done |
| 5 | Correlation Heatmap | Done |
| 6 | Feature Scaling | Done |
| 7 | Model Training & Evaluation | Done |
| 8 | Feature Engineering | Done |
| 9 | Model Saved for Deployment | Done |